In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from collections import Counter

# -------------------- Data Preparation -------------------- #
team_mapping = {
    'Man Utd': 'Manchester Utd', 'Man United': 'Manchester Utd',
    'Man City': 'Manchester City', 'Newcastle Utd': 'Newcastle United',
    'Newcastle Ut': 'Newcastle United', "Nott'ham Forest": 'Nottingham Forest',
    'Paris S-G': 'Paris Saint-Germain', 'Inter Milan': 'Inter',
    'Spurs': 'Tottenham', 'West Ham Utd': 'West Ham United'
}

def clean_numeric_values(value):
    """Convert numbers with commas and handle missing values"""
    if isinstance(value, str):
        return float(value.replace(',', '')) if value.replace(',', '').isdigit() else np.nan
    return value

def standardize_team_names(df, column_name):
    df[column_name] = df[column_name].replace(team_mapping).str.strip()
    return df

def load_and_preprocess_data():
    # Load data
    stats_df = pd.read_csv('Combined_Leagues_Stats.csv')
    fixtures_df = pd.read_csv('Fixture_Results.csv')

    # Clean numeric columns
    numeric_cols = ['progressive_carries', 'progressive_passes', 'xg', 'npxg', 
                   'xg_assist', 'npxg_xg_assist', 'goals_per90', 'assists_per90',
                   'goals_assists_per90', 'goals_pens_per90', 'xg_per90',
                   'xg_assist_per90', 'npxg_per90', 'Home_xG', 'Away_xG']
    
    for df in [stats_df, fixtures_df]:
        for col in numeric_cols:
            if col in df.columns:
                df[col] = df[col].apply(clean_numeric_values)

    # Standardize team names
    stats_df = standardize_team_names(stats_df, 'team')
    fixtures_df = standardize_team_names(fixtures_df, 'Home_Team')
    fixtures_df = standardize_team_names(fixtures_df, 'Away_Team')

    # Feature engineering
    stats_df['total_progression'] = stats_df['progressive_carries'] + stats_df['progressive_passes']

    # Select and order features
    feature_columns = [
        'xg', 'npxg', 'xg_assist', 'npxg_xg_assist',
        'total_progression', 'goals_per90', 'assists_per90', 
        'goals_assists_per90', 'goals_pens_per90', 'xg_per90',
        'xg_assist_per90', 'npxg_per90'
    ]

    # Handle missing values
    imputer = SimpleImputer(strategy='median')
    stats_df[feature_columns] = imputer.fit_transform(stats_df[feature_columns])
    
    # Create team stats mapping
    team_stats = stats_df.set_index('team')[feature_columns].to_dict('index')
    
    return team_stats, fixtures_df

# -------------------- Model Architecture -------------------- #
class FootballPredictor(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout=0.3):
        super().__init__()
        self.shared_base = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.class_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU(),
            nn.Linear(hidden_dim//2, 3)
        )
        self.reg_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU(),
            nn.Linear(hidden_dim//2, 2)
        )

    def forward(self, x):
        x = self.shared_base(x)
        return self.class_head(x), self.reg_head(x)

# -------------------- Training Utilities -------------------- #
class EarlyStopper:
    def __init__(self, patience=10, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_val_loss = float('inf')

    def __call__(self, val_loss):
        if val_loss < self.min_val_loss - self.min_delta:
            self.min_val_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
        return self.counter >= self.patience

def get_class_weights(y_class):
    class_counts = Counter(y_class)
    weights = [1/class_counts[i] for i in range(3)]
    return torch.tensor(weights, dtype=torch.float32)

# -------------------- Main Execution -------------------- #
if __name__ == "__main__":
    # Load and preprocess data
    team_stats, fixtures_df = load_and_preprocess_data()
    feature_columns = list(next(iter(team_stats.values())).keys())

    # Feature extraction
    def get_features(row):
        home_features = list(team_stats.get(row['Home_Team'].strip(), {}).values()) or [0]*len(feature_columns)
        away_features = list(team_stats.get(row['Away_Team'].strip(), {}).values()) or [0]*len(feature_columns)
        return home_features + away_features

    fixtures_df['features'] = fixtures_df.apply(get_features, axis=1)
    fixtures_df['result'] = fixtures_df.apply(
        lambda row: 2 if row['Home_Score'] > row['Away_Score'] else 1 if row['Home_Score'] == row['Away_Score'] else 0, 
        axis=1
    )

    # Prepare datasets
    X = np.array(fixtures_df['features'].tolist())
    y_class = fixtures_df['result'].values
    y_reg = fixtures_df[['Home_xG', 'Away_xG']].values
    
    X_train, X_test, y_class_train, y_class_test, y_reg_train, y_reg_test = train_test_split(
        X, y_class, y_reg, test_size=0.2, random_state=42
    )

    # Normalization
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Dataset class
    class FootballDataset(Dataset):
        def __init__(self, X, y_class, y_reg):
            self.X = torch.tensor(X, dtype=torch.float32)
            self.y_class = torch.tensor(y_class, dtype=torch.long)
            self.y_reg = torch.tensor(y_reg, dtype=torch.float32)
        
        def __len__(self): return len(self.X)
        def __getitem__(self, idx): return self.X[idx], self.y_class[idx], self.y_reg[idx]

    train_dataset = FootballDataset(X_train, y_class_train, y_reg_train)
    test_dataset = FootballDataset(X_test, y_class_test, y_reg_test)
    
    # Initialize model
    input_dim = X_train.shape[1]
    model = FootballPredictor(input_dim)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5)
    early_stopper = EarlyStopper(patience=10)
    
    # Loss functions
    class_weights = get_class_weights(y_class_train)
    criterion_class = nn.CrossEntropyLoss(weight=class_weights)
    criterion_reg = nn.MSELoss()

    # Training loop
    for epoch in range(100):
        model.train()
        total_loss = 0
        for X_batch, y_class_batch, y_reg_batch in DataLoader(train_dataset, batch_size=32, shuffle=True):
            optimizer.zero_grad()
            class_out, reg_out = model(X_batch)
            loss = 0.7*criterion_class(class_out, y_class_batch) + 0.3*criterion_reg(reg_out, y_reg_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_class_batch, y_reg_batch in DataLoader(test_dataset, batch_size=32):
                class_out, reg_out = model(X_batch)
                val_loss += 0.7*criterion_class(class_out, y_class_batch) + 0.3*criterion_reg(reg_out, y_reg_batch)
        
        avg_val_loss = val_loss / len(test_dataset)
        scheduler.step(avg_val_loss)
        
        print(f"Epoch {epoch+1}: Train Loss: {total_loss/len(train_dataset):.4f}, Val Loss: {avg_val_loss:.4f}")
        
        if early_stopper(avg_val_loss):
            print("Early stopping triggered")
            break

    # Prediction function
    def predict_fixtures(model, scaler, fixtures_csv_path):
        new_fixtures = pd.read_csv(fixtures_csv_path)
        new_fixtures = standardize_team_names(new_fixtures, 'Home_Team')
        new_fixtures = standardize_team_names(new_fixtures, 'Away_Team')
        
        new_fixtures['features'] = new_fixtures.apply(get_features, axis=1)
        X_new = scaler.transform(np.array(new_fixtures['features'].tolist()))
        X_new = torch.tensor(X_new, dtype=torch.float32)
        
        model.eval()
        with torch.no_grad():
            class_logits, reg_preds = model(X_new)
            class_probs = torch.softmax(class_logits, dim=1).numpy()
        
        return pd.DataFrame({
            'Home_Team': new_fixtures['Home_Team'],
            'Away_Team': new_fixtures['Away_Team'],
            'Home_Win_Prob': class_probs[:, 2],
            'Draw_Prob': class_probs[:, 1],
            'Away_Win_Prob': class_probs[:, 0],
            'Predicted_Home_xG': reg_preds[:, 0].numpy(),
            'Predicted_Away_xG': reg_preds[:, 1].numpy()
        })

Epoch 1: Train Loss: 0.0333, Val Loss: 0.0298
Epoch 2: Train Loss: 0.0297, Val Loss: 0.0289
Epoch 3: Train Loss: 0.0282, Val Loss: 0.0285
Epoch 4: Train Loss: 0.0281, Val Loss: 0.0284
Epoch 5: Train Loss: 0.0279, Val Loss: 0.0283
Epoch 6: Train Loss: 0.0276, Val Loss: 0.0284
Epoch 7: Train Loss: 0.0282, Val Loss: 0.0285
Epoch 8: Train Loss: 0.0277, Val Loss: 0.0282
Epoch 9: Train Loss: 0.0272, Val Loss: 0.0286
Epoch 10: Train Loss: 0.0280, Val Loss: 0.0282
Epoch 11: Train Loss: 0.0277, Val Loss: 0.0284
Epoch 12: Train Loss: 0.0273, Val Loss: 0.0281
Epoch 13: Train Loss: 0.0271, Val Loss: 0.0283
Early stopping triggered


In [2]:
predictions = predict_fixtures(model, scaler, 'fixtures.csv')
print(predictions[['Home_Team', 'Away_Team', 'Home_Win_Prob', 'Draw_Prob', 'Away_Win_Prob', 'Predicted_Home_xG', 'Predicted_Away_xG']])

       Home_Team        Away_Team  Home_Win_Prob  Draw_Prob  Away_Win_Prob  \
0      West Brom   Sheffield Weds       0.312095   0.373182       0.314724   
1     Sunderland          Watford       0.387620   0.396574       0.215806   
2   Norwich City     Derby County       0.566013   0.323003       0.110984   
3  Sheffield Utd       Portsmouth       0.464895   0.373873       0.161232   
4     Celta Vigo            Betis       0.396931   0.388262       0.214807   
5  Athletic Club           Girona       0.433694   0.374110       0.192196   
6     Las Palmas       Villarreal       0.147132   0.281494       0.571373   
7    Real Madrid  Atlético Madrid       0.421348   0.383306       0.195345   

   Predicted_Home_xG  Predicted_Away_xG  
0           1.130215           0.954242  
1           1.274670           0.907039  
2           1.665034           0.561216  
3           1.510711           0.773552  
4           1.370266           1.096977  
5           1.432112           0.926325  
6  